### Завантаження даних

Цей етап імпортує необхідні бібліотеки та завантажує набір даних для аналізу.

In [1]:
import pandas as pd
import pyreadstat
import statsmodels.api as sm

df, meta = pyreadstat.read_sav('data/Kyiv1991.SAV')

### Підготовка даних

На цьому етапі відбираються дві змінні політичного типу та дві змінні матеріального типу, видаляються пропущені значення та формується дихотомічна залежна змінна разом із матрицею незалежних факторів.

In [2]:
selected_cols = ['V60', 'V36', 'V84', 'V4', 'V21']
df_clean = df[selected_cols].dropna()

y = (df_clean['V60'] == 1).astype(int)
X = df_clean[['V36', 'V84', 'V4', 'V21']]
X = sm.add_constant(X)

### Побудова моделі логістичної регресії

Тут створюється та навчається модель бінарної логістичної регресії, а також виводяться детальні результати оцінки якості моделі та коефіцієнти.

In [3]:
model = sm.Logit(y, X)
result = model.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.605007
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                    V60   No. Observations:                  418
Model:                          Logit   Df Residuals:                      413
Method:                           MLE   Df Model:                            4
Date:                Mon, 25 May 2026   Pseudo R-squ.:                 0.07510
Time:                        18:08:18   Log-Likelihood:                -252.89
converged:                       True   LL-Null:                       -273.43
Covariance Type:            nonrobust   LLR p-value:                 2.602e-08
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.0493      0.558      1.879      0.060      -0.045       2.144
V36           -0.4512      0.

### Рівняння та автоматизовані висновки

Цей блок формує кінцеве рівняння регресії та програмно аналізує напрямок і значущість впливу кожного фактору на основі отриманих p-значень та коефіцієнтів.

In [4]:
print(f"Logit(P) = {result.params['const']:.4f} + {result.params['V36']:.4f}*V36 + {result.params['V84']:.4f}*V84 + {result.params['V4']:.4f}*V4 + {result.params['V21']:.4f}*V21")
print(f"Pseudo R-squared: {result.prsquared:.4f}\n")

for col in ['V36', 'V84', 'V4', 'V21']:
    coef = result.params[col]
    pval = result.pvalues[col]
    if pval < 0.05:
        direction = "позитивно" if coef > 0 else "негативно"
        print(f"Фактор {col} впливає {direction}")
    else:
        print(f"Фактор {col} не має статистично значущого впливу")

Logit(P) = 1.0493 + -0.4512*V36 + -0.2607*V84 + -0.0032*V4 + -0.0462*V21
Pseudo R-squared: 0.0751

Фактор V36 впливає негативно
Фактор V84 не має статистично значущого впливу
Фактор V4 не має статистично значущого впливу
Фактор V21 не має статистично значущого впливу
